### Лабораторная работа 5
#### Клименко В.А.

##### Импорт и загрузка данных

In [ ]:
import os
import json
import re
from pprint import pprint

import spacy
from spacy import displacy
import pandas as pd
from collections import defaultdict

# Настройка отображения
pd.set_option("display.max_colwidth", None)

texts_data = {}
gt_data = {}

news_dir = "./lab5_data/news/"
gt_path = "./lab5_data/ground_truth.json"

if os.path.exists(news_dir):
    for fname in sorted(os.listdir(news_dir)):
        if fname.endswith(".txt"):
            with open(
                os.path.join(news_dir, fname), "r", encoding="utf-8"
            ) as f:
                texts_data[fname] = f.read()

if os.path.exists(gt_path):
    with open(gt_path, "r", encoding="utf-8") as f:
        gt_data = json.load(f)

print(f"✅ Загружено {len(texts_data)} текстов.")
print("Пример ground truth для news_01.txt:")
pprint(gt_data['news_01.txt'], sort_dicts=True)

##### Подход 1: Регулярные выражения (Regex)

In [ ]:
def extract_with_regex(text):
    entities = {"DATE": [], "MONEY": []}

    date_pattern = r"\b\d{1,2}\s+[а-яА-ЯёЁ]+(?:\s+\d{4})?\b"
    money_pattern = r"\b\d+(?:[\.,]\d+)?\s*(?:млн|тыс|млрд)?\s*(?:рублей|рубля|руб|долларов|доллара|евро|\$)\b"

    for match in re.finditer(date_pattern, text, re.IGNORECASE):
        entities["DATE"].append(match.group(0).strip())

    for match in re.finditer(money_pattern, text, re.IGNORECASE):
        entities["MONEY"].append(match.group(0).strip())

    return entities


# Проверка на первом тексте
test_text = texts_data["news_01.txt"]
print(f"Текст: {test_text[:100]}...")
print(f"Regex нашел: {extract_with_regex(test_text)}")

##### Подход 2: Машинное обучение (spaCy)

In [ ]:
nlp = spacy.load("ru_core_news_sm")


def extract_with_spacy(text):
    entities = defaultdict(list)
    doc = nlp(text)
    for ent in doc.ents:
        entities[ent.label_].append(ent.text)
    return dict(entities)


# Проверка
print(f"spaCy нашел: {extract_with_spacy(test_text)}")

# ВАУ-ЭФФЕКТ: Визуализация
doc = nlp(test_text)
displacy.render(doc, style="ent", jupyter=True)

##### Оценка качества (Метрики)

In [ ]:
def calculate_metrics(predicted, ground_truth):
    # Вспомогательная функция для нормализации текста сущностей
    def normalize(s):
        return s.lower().strip().rstrip(".,!?:;\"'«»")

    # Собираем все сущности из словарей в единые множества (упрощенно по всем типам сразу)
    pred_set = set()
    for ent_list in predicted.values():
        for ent in ent_list:
            pred_set.add(normalize(ent))

    gt_set = set()
    for ent_list in ground_truth.values():
        for ent in ent_list:
            gt_set.add(normalize(ent))

    # Пересечение (найденное верное)
    intersection = pred_set.intersection(gt_set)

    precision = len(intersection) / len(pred_set) if len(pred_set) > 0 else 0.0
    recall = len(intersection) / len(gt_set) if len(gt_set) > 0 else 0.0
    f1 = (
        (2 * precision * recall) / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )

    return round(precision, 3), round(recall, 3), round(f1, 3)


# Сравнение на всех текстах
results = []
for fname in texts_data.keys():
    text = texts_data[fname]
    gt = gt_data[fname]
    pred_regex = extract_with_regex(text)
    pred_spacy = extract_with_spacy(text)
    # Считаем метрики (упрощенно, по всем типам сущностей сразу)
    p_reg, r_reg, f1_reg = calculate_metrics(pred_regex, gt)
    p_spc, r_spc, f1_spc = calculate_metrics(pred_spacy, gt)

    results.append(
        {
            "Файл": fname,
            "Regex_P": p_reg,
            "Regex_R": r_reg,
            "Regex_F1": f1_reg,
            "spaCy_P": p_spc,
            "spaCy_R": r_spc,
            "spaCy_F1": f1_spc,
        }
    )
print("📊 СРАВНЕНИЕ МЕТОДОВ:")
df_results = pd.DataFrame(results)
display(df_results)

##### Анализ ошибок (Самое важное!)

In [ ]:

print("🔍 АНАЛИЗ ОШИБОК:")

# Найдите текст, где Regex нашел дату неправильно (например, "5 ноября" контексте спорта), или где spaCy не нашел организацию


def get_normalized_set(ent_dict):
    return {
        ent.lower().strip().rstrip(".,!?:;\"'«»")
        for ent_list in ent_dict.values()
        for ent in ent_list
    }


for fname in list(texts_data.keys())[:5]:  # Просмотрим первые 5 файлов
    text = texts_data[fname]
    gt = gt_data[fname]
    pred_regex = extract_with_regex(text)
    pred_spacy = extract_with_spacy(text)

    gt_set = get_normalized_set(gt)
    reg_set = get_normalized_set(pred_regex)
    spc_set = get_normalized_set(pred_spacy)

    print(f"=== Файл: {fname} ===")
    print(f"Текст: {text}")

    print("Эталон (GT): ", end="")
    pprint(gt, sort_dicts=True)

    print(f"Нашел Regex: ", end="")
    pprint(pred_regex, sort_dicts=True)

    print(f"Нашел spaCy: ", end="")
    pprint(pred_spacy, sort_dicts=True)

    print(f"\t-> Regex пропустил (но нашел spaCy): {gt_set.intersection(spc_set) - reg_set}")
    print(f"\t-> spaCy пропустил (но нашел Regex): {gt_set.intersection(reg_set) - spc_set}")
    print(f"\t-> Ложные срабатывания Regex (нет в GT): {reg_set - gt_set}")
    print(f"\t-> Ложные срабатывания spaCy (нет в GT): {spc_set - gt_set}")
    print()

##### Итоговые выводы

In [ ]:
print("Какой метод (Regex или spaCy) показал более высокий F1-score?")
display(df_results.style.highlight_max(axis=1, subset=["Regex_F1", "spaCy_F1"], color="green"))

print("Почему Regex плохо справляется с извлечением имен (PER) и организаций (ORG)?")
print(
    """
    Имена и организации не имеют жесткой символьной структуры:
    могут состоять из любых слов, склоняться по падежам
    и совпадать с обычными существительными.
    """
)

print("Приведите пример, где spaCy ошибся (ложноположительное срабатывание). Почему?")
print(
    """
    spaCy может ошибочно выделить слово с заглавной буквы в начале предложения
    или название в кавычках как организацию (ORG) или персону (PER), либо захватить соседнее слово,
    потому что модель опирается на статистические паттерны контекста и регистр букв.
    """
)

print("В каких реальных задачах лучше использовать Regex, а в каких ML?")
print(
    """
    Regex лучше использовать для сущностей со строгим форматом:
    даты,суммы денег, номера телефонов, email и т.д.
    ML необходим для извлечения смысловых и контекстно-зависимых сущностей в свободном тексте:
    названия компаний, географические объекты, должности.
    """
)